# MCP Connectivity Tests

Validate connectivity and tool discovery for the remote MCP servers using the official MCP Python SDK.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

Project Root: c:\Users\praya\Desktop\Tarvel_Agent\trip_planner


In [2]:
from config.settings import settings

SERVERS = [
    {"name": "Kiwi MCP", "url": settings.kiwi_mcp_server_url},
    {"name": "Gribstream MCP", "url": settings.gribstream_mcp_server_url},
    {"name": "Agentorist MCP", "url": settings.agentorist_mcp_server_url},
]

for server in SERVERS:
    print(f"{server['name']}: {server['url']}")

Kiwi MCP: https://mcp.kiwi.com
Gribstream MCP: https://gribstream.com/mcp
Agentorist MCP: https://mcp.agentorist.com/mcp


In [3]:
import asyncio

from mcp import ClientSession
from mcp.client.sse import sse_client
from mcp.client.streamable_http import streamable_http_client


async def list_tools_for_url(url: str) -> list[str]:
    normalized = url.rstrip("/")
    if normalized.endswith("/sse"):
        async with sse_client(url) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [tool.name for tool in tools.tools]

    async with streamable_http_client(url) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await session.list_tools()
            return [tool.name for tool in tools.tools]


async def test_server(server: dict) -> dict:
    result = {
        "server": server["name"],
        "url": server["url"],
        "status": "unknown",
        "tools": [],
    }

    try:
        tools = await list_tools_for_url(server["url"])
        result["status"] = "connected"
        result["tools"] = tools
    except Exception as exc:
        result["status"] = f"failed: {exc}"

    return result


async def run_tests() -> list[dict]:
    results = []
    for server in SERVERS:
        results.append(await test_server(server))
    return results

In [4]:
results = await run_tests()

for result in results:
    print("=" * 60)
    print("Server:", result["server"])
    print("URL:", result["url"])
    print("Connection:", result["status"])
    print("Available tools:", result["tools"])

Server: Kiwi MCP
URL: https://mcp.kiwi.com
Connection: connected
Available tools: ['search-flight', 'feedback-to-devs']
Server: Gribstream MCP
URL: https://gribstream.com/mcp
Connection: failed: unhandled errors in a TaskGroup (1 sub-exception)
Available tools: []
Server: Agentorist MCP
URL: https://mcp.agentorist.com/mcp
Connection: connected
Available tools: ['list_verticals', 'search', 'search_all', 'find_options', 'book', 'list_venues', 'request_unsupported_booking']


In [12]:
import asyncio
import traceback

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async def test_kiwi_direct():
    try:
        async with streamable_http_client("https://mcp.kiwi.com") as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()

                result = await session.call_tool(
                    "search-flight",
                    {
                        "flyFrom": "Delhi",
                        "flyTo": "Mumbai",
                        "departureDate": "2026-08-15"
                    }
                )

                print(result)

    except Exception as e:
        print("TYPE:", type(e))
        print("ERROR:", e)

        if hasattr(e, "exceptions"):
            print("\nNested exceptions:")
            for i, ex in enumerate(e.exceptions, 1):
                print(f"\n--- Exception {i} ---")
                print(type(ex))
                print(ex)
                traceback.print_exception(type(ex), ex, ex.__traceback__)

        traceback.print_exception(type(e), e, e.__traceback__)

await test_kiwi_direct()

TYPE: <class 'ExceptionGroup'>
ERROR: unhandled errors in a TaskGroup (1 sub-exception)

Nested exceptions:

--- Exception 1 ---
<class 'ExceptionGroup'>
unhandled errors in a TaskGroup (1 sub-exception)


  + Exception Group Traceback (most recent call last):
  |   File "c:\Users\praya\Desktop\Tarvel_Agent\.venv\Lib\site-packages\mcp\client\streamable_http.py", line 670, in streamable_http_client
  |     yield (
  |     ...<3 lines>...
  |     )
  |   File "C:\Users\praya\AppData\Local\Temp\ipykernel_52696\1572464912.py", line 14, in test_kiwi_direct
  |     async with ClientSession(read_stream, write_stream) as session:
  |                ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "c:\Users\praya\Desktop\Tarvel_Agent\.venv\Lib\site-packages\mcp\shared\session.py", line 238, in __aexit__
  |     return await self._task_group.__aexit__(exc_type, exc_val, exc_tb)
  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "c:\Users\praya\Desktop\Tarvel_Agent\.venv\Lib\site-packages\anyio\_backends\_asyncio.py", line 799, in __aexit__
  |     raise BaseExceptionGroup(
  |         "unhandled errors in a TaskGroup", self._exceptions
  |     ) from None
  | 

In [19]:
import asyncio
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async def test_kiwi():
    async with streamable_http_client("https://mcp.kiwi.com") as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()

            result = await session.call_tool(
                "search-flight",
                {
                    "flyFrom": "DEL",
                    "flyTo": "BOM",
                    "departureDate": "15/08/2026",
                    "curr": "INR",
                    "sortBy": "price"
                }
            )

            print(result)

await test_kiwi()

meta=None content=[TextContent(type='text', text='[\n  {\n    "flyFrom": "DEL",\n    "flyTo": "BOM",\n    "cityFrom": "New Delhi",\n    "cityTo": "Mumbai",\n    "departure": {\n      "utc": "2026-08-14T22:30:00.000Z",\n      "local": "2026-08-15T04:00:00.000"\n    },\n    "arrival": {\n      "utc": "2026-08-15T00:45:00.000Z",\n      "local": "2026-08-15T06:15:00.000"\n    },\n    "totalDurationInSeconds": 8100,\n    "durationInSeconds": 8100,\n    "price": 7085,\n    "deepLink": "https://on.kiwi.com/0nSVe2",\n    "currency": "INR"\n  },\n  {\n    "flyFrom": "DEL",\n    "flyTo": "BOM",\n    "cityFrom": "New Delhi",\n    "cityTo": "Mumbai",\n    "departure": {\n      "utc": "2026-08-15T03:00:00.000Z",\n      "local": "2026-08-15T08:30:00.000"\n    },\n    "arrival": {\n      "utc": "2026-08-15T05:15:00.000Z",\n      "local": "2026-08-15T10:45:00.000"\n    },\n    "totalDurationInSeconds": 8100,\n    "durationInSeconds": 8100,\n    "price": 7704,\n    "deepLink": "https://on.kiwi.com/GURo